# LeetCode #1195: Fizz Buzz Multithreaded

https://leetcode.com/problems/fizz-buzz-multithreaded/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin loop on shared counter | Wastes CPU; races on counter |
| **Optimal: Four Semaphores + Shared Counter ★** | One gate per output type | Each thread wakes only when its turn matches the counter |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Four Semaphores + Shared Counter ★
Maintain a shared counter. A dispatcher (or the `number` thread) increments it and signals the correct semaphore each turn: `fizzbuzzSem` if divisible by 15, `fizzSem` if by 3, `buzzSem` if by 5, otherwise `numSem`. Each worker thread loops, acquires its semaphore, performs its output, and signals the dispatcher to advance.

**Constraints:**
* Four threads: `fizz()`, `buzz()`, `fizzbuzz()`, `number()`
* Shared counter from 1 to n
* `1 <= n <= 50`


## Solutions
### C#

In [ ]:
using System.Threading;

public class FizzBuzz
{
    private readonly int _n;
    private int _i = 1;
    private readonly SemaphoreSlim _numSem  = new SemaphoreSlim(1, 1);
    private readonly SemaphoreSlim _fizzSem = new SemaphoreSlim(0, 1);
    private readonly SemaphoreSlim _buzzSem = new SemaphoreSlim(0, 1);
    private readonly SemaphoreSlim _fbSem   = new SemaphoreSlim(0, 1);
    private readonly SemaphoreSlim _doneSem = new SemaphoreSlim(0, 1);

    public FizzBuzz(int n) => _n = n;

    private void Advance()
    {
        _i++;
        if (_i > _n) { _fizzSem.Release(); _buzzSem.Release(); _fbSem.Release(); _numSem.Release(); return; }
        if (_i % 15 == 0) _fbSem.Release();
        else if (_i % 3 == 0) _fizzSem.Release();
        else if (_i % 5 == 0) _buzzSem.Release();
        else _numSem.Release();
    }

    public void Number(Action<int> printNumber)
    {
        while (true)
        {
            _numSem.Wait();
            if (_i > _n) return;
            printNumber(_i);
            Advance();
        }
    }

    public void Fizz(Action printFizz)
    {
        while (true)
        {
            _fizzSem.Wait();
            if (_i > _n) return;
            printFizz();
            Advance();
        }
    }

    public void Buzz(Action printBuzz)
    {
        while (true)
        {
            _buzzSem.Wait();
            if (_i > _n) return;
            printBuzz();
            Advance();
        }
    }

    public void FizzBuzz(Action printFizzBuzz)
    {
        while (true)
        {
            _fbSem.Wait();
            if (_i > _n) return;
            printFizzBuzz();
            Advance();
        }
    }
}

### Python

In [ ]:
import threading

class FizzBuzz:
    def __init__(self, n: int):
        self.n = n
        self.i = 1
        self.num_sem  = threading.Semaphore(1)
        self.fizz_sem = threading.Semaphore(0)
        self.buzz_sem = threading.Semaphore(0)
        self.fb_sem   = threading.Semaphore(0)

    def _advance(self):
        self.i += 1
        if self.i > self.n:
            # wake all so they can exit
            self.fizz_sem.release(); self.buzz_sem.release()
            self.fb_sem.release();   self.num_sem.release()
            return
        if   self.i % 15 == 0: self.fb_sem.release()
        elif self.i % 3  == 0: self.fizz_sem.release()
        elif self.i % 5  == 0: self.buzz_sem.release()
        else:                  self.num_sem.release()

    def number(self, printNumber) -> None:
        while True:
            self.num_sem.acquire()
            if self.i > self.n: return
            printNumber(self.i)
            self._advance()

    def fizz(self, printFizz) -> None:
        while True:
            self.fizz_sem.acquire()
            if self.i > self.n: return
            printFizz()
            self._advance()

    def buzz(self, printBuzz) -> None:
        while True:
            self.buzz_sem.acquire()
            if self.i > self.n: return
            printBuzz()
            self._advance()

    def fizzbuzz(self, printFizzBuzz) -> None:
        while True:
            self.fb_sem.acquire()
            if self.i > self.n: return
            printFizzBuzz()
            self._advance()

### Go

In [ ]:
package main

type FizzBuzz struct {
	n    int
	i    int
	num  chan struct{}
	fizz chan struct{}
	buzz chan struct{}
	fb   chan struct{}
}

func NewFizzBuzz(n int) *FizzBuzz {
	f := &FizzBuzz{
		n:    n, i: 1,
		num:  make(chan struct{}, 1),
		fizz: make(chan struct{}, 1),
		buzz: make(chan struct{}, 1),
		fb:   make(chan struct{}, 1),
	}
	f.num <- struct{}{} // 1 is not divisible by 3 or 5
	return f
}

func (f *FizzBuzz) advance() {
	f.i++
	if f.i > f.n {
		f.num <- struct{}{}; f.fizz <- struct{}{}; f.buzz <- struct{}{}; f.fb <- struct{}{}
		return
	}
	switch {
	case f.i%15 == 0: f.fb   <- struct{}{}
	case f.i%3  == 0: f.fizz <- struct{}{}
	case f.i%5  == 0: f.buzz <- struct{}{}
	default:          f.num  <- struct{}{}
	}
}

func (f *FizzBuzz) Number(print func(int)) {
	for { <-f.num; if f.i > f.n { return }; print(f.i); f.advance() }
}
func (f *FizzBuzz) Fizz(print func()) {
	for { <-f.fizz; if f.i > f.n { return }; print(); f.advance() }
}
func (f *FizzBuzz) Buzz(print func()) {
	for { <-f.buzz; if f.i > f.n { return }; print(); f.advance() }
}
func (f *FizzBuzz) FizzBuzz(print func()) {
	for { <-f.fb; if f.i > f.n { return }; print(); f.advance() }
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex, Condvar};

struct FizzBuzz {
    n: i32,
    i: Mutex<i32>,
    cv: Condvar,
}

impl FizzBuzz {
    fn new(n: i32) -> Arc<Self> {
        Arc::new(FizzBuzz { n, i: Mutex::new(1), cv: Condvar::new() })
    }

    fn number(&self, print_number: impl Fn(i32)) {
        loop {
            let mut i = self.i.lock().unwrap();
            while *i <= self.n && (*i % 3 == 0 || *i % 5 == 0) { i = self.cv.wait(i).unwrap(); }
            if *i > self.n { return; }
            print_number(*i); *i += 1; self.cv.notify_all();
        }
    }

    fn fizz(&self, print_fizz: impl Fn()) {
        loop {
            let mut i = self.i.lock().unwrap();
            while *i <= self.n && !(*i % 3 == 0 && *i % 5 != 0) { i = self.cv.wait(i).unwrap(); }
            if *i > self.n { return; }
            print_fizz(); *i += 1; self.cv.notify_all();
        }
    }

    fn buzz(&self, print_buzz: impl Fn()) {
        loop {
            let mut i = self.i.lock().unwrap();
            while *i <= self.n && !(*i % 5 == 0 && *i % 3 != 0) { i = self.cv.wait(i).unwrap(); }
            if *i > self.n { return; }
            print_buzz(); *i += 1; self.cv.notify_all();
        }
    }

    fn fizzbuzz(&self, print_fizzbuzz: impl Fn()) {
        loop {
            let mut i = self.i.lock().unwrap();
            while *i <= self.n && *i % 15 != 0 { i = self.cv.wait(i).unwrap(); }
            if *i > self.n { return; }
            print_fizzbuzz(); *i += 1; self.cv.notify_all();
        }
    }
}

## Concurrency Scenarios

1. **i = 15 (FizzBuzz)**: Only `fbSem` is signaled; fizz, buzz, and number threads remain blocked — correct routing.
2. **All four threads start before dispatcher**: Number starts with `numSem(1)` so number thread proceeds for i=1; others wait on their 0-count semaphores.
3. **n = 3 (outputs: 1, 2, Fizz)**: Number→1, Number→2, Fizz→3; buzz and fizzbuzz threads wake once at end due to termination broadcast and exit.
4. **Termination broadcast**: When `i > n`, all four semaphores are released once so all blocked threads can detect the end and return.
5. **Shared counter with no races**: Only the thread that acquired its semaphore can run `Advance()` — the counter is effectively single-writer at any moment.
